# धडा 03 - एजंटिक डिझाइन पॅटर्न्स

या धड्यात, आपण कार्यक्षम AI एजंट तयार करण्यासाठी तीन मूलभूत डिझाइन पॅटर्न्सचा आढावा घेणार आहोत:

1. **स्पष्ट एजंट सूचना** — एजंटच्या वर्तनाला मार्गदर्शन करणारे नेमके, भूमिका-निर्धारण करणारे प्रॉम्प्ट तयार करणे
2. **पायडांटिक मॉडेल्ससह संरचित आउटपुट** — एजंट्सकडून आशारीय, पडताळणीकृत डेटा मिळवणे सुनिश्चित करणे
3. **एकल जबाबदारी एजंट्स** — लक्ष केंद्रित करणारे एजंट डिझाइन करणे जे प्रत्येकी एक गोष्ट चांगल्या प्रकारे करतात

आपण प्रत्येक पॅटर्नचा वापर **भटकंती गंतव्य शिफारस करणारा** परिदृश्यात करू, जे हळूहळू असे सिस्टम तयार करेल जे गंतव्ये सुचवू शकेल, उपलब्धता तपासेल आणि लॉजिस्टिक्स हाताळेल.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## नमुना 1: स्पष्ट एजंट सूचना

सर्वाधिक प्रभावी नमुना देखील सर्वात सोपा आहे: आपल्या एजंटसाठी स्पष्ट, तपशीलवार सूचना लिहिणे.

चांगल्या सूचनांमध्ये समाविष्ट आहे:
- **कोण** एजंट आहे (व्यक्तिमत्व आणि टोन)
- **काय** करावे (टप्प्याटप्प्याने जबाबदाऱ्या)
- **कसे** वागावे (मर्यादा आणि शैली)

खाली, आम्ही प्रवास कन्सिएर्ज एजंट तयार करतो ज्यामध्ये प्रत्येक प्रतिसादावर स्पष्ट सूचना असतात ज्या तो निर्माण करतो.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

The key to enforcement is passing `response_format` when we run the agent. This forces the
model to return a validated `TravelRecommendations` object (available on `response.value`)
instead of free-form text. The `get_destination_details` tool also returns a typed
`DestinationRecommendation`, so the data stays structured end to end.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## नमुना 3: एकच जबाबदारी एजंट

गुंतागुंतीच्या कामांना अनेक लक्ष केंद्रित एजंटमध्ये विभाजित केल्याने फायदा होतो, प्रत्येक एजंटची एकच जबाबदारी असते:

- एक **डेस्टिनेशन एक्सपर्ट** जो ठिकाणे आणि उपलब्धतेबद्दल माहिती असतो
- एक **लॉजिस्टिक्स प्लॅनर** जो फ्लाइट्स, हॉटेल्स आणि प्रवासयोजना हाताळतो

हे सॉफ्टवेअर अभियांत्रिकीच्या *सेपरेशन ऑफ कॉन्सर्न्स* तत्त्वाचे प्रतिबिंब आहे — प्रत्येक एजंट स्वतंत्रपणे चाचणी, देखभाल आणि सुधारणा करणे सोपे आहे.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

या धड्यात आपल्याने ट्रॅव्हल शिफारस करणाऱ्या परिदृश्यावर तीन एजंटिक डिझाइन पॅटर्न लागू केले:

| पॅटर्न | मुख्य कल्पना | फायदे |
|---|---|---|
| **स्पष्ट सूचना** | व्यक्तिमत्व, जबाबदाऱ्या आणि बंधने पूर्वनिर्धारित करा | सुसंगत, ब्रँडच्या अनुरूप एजंट वागणूक |
| **संरचित आउटपुट** | प्रतिसादाच्या स्वरूपासाठी Pydantic मॉडेल वापरा | प्रमाणित, मशीन-वाचनीय निकाल |
| **एकट्या जबाबदारी** | प्रत्येक एजंटला एक केंद्रीत काम द्या | परीक्षा, देखभाल आणि संयोजन करणे सोपे |

हे पॅटर्न नैसर्गिक रीतीने एकत्र होतात — आपण स्पष्ट सूचना आणि संरचित आउटपुट एका एकट्या जबाबदारी असलेल्या एजंटमध्ये एकत्र करून मजबूत, उत्पादन-तयार प्रणाली तयार करू शकता.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
